In [5]:
"""
2. Составить такой же поисковик с одним агентом и добавить к нему planning, memory и knowledge
"""

import os
from dotenv import load_dotenv
from agno.agent import Agent
from agno.tools.duckduckgo import DuckDuckGoTools
from agno.tools.reasoning import ReasoningTools
from agno.db.sqlite import SqliteDb
from agno.models.openai.like import OpenAILike
from agno.knowledge.knowledge import Knowledge
from agno.vectordb.lancedb import LanceDb, SearchType
from agno.knowledge.embedder.openai import OpenAIEmbedder

load_dotenv()

model = OpenAILike(
    id=os.getenv('MODEL_ID'),
    base_url=os.getenv('OPENROUTER_BASE_URL'),
    api_key=os.getenv('OPENROUTER_API_KEY') 
)

db = SqliteDb(
    session_table="agent_sessions",
    db_file="tmp/single_agent_search.db"
)

knowledge_base = Knowledge(
    vector_db=LanceDb(
        uri="tmp/lancedb",
        table_name="search_knowledge",
        search_type=SearchType.vector,
        embedder=OpenAIEmbedder(
            id="text-embedding-3-small",
        ),
    ),
)

os.makedirs("tmp/knowledge", exist_ok=True)

horror_films_content = """Сияние (1980) - психологический хоррор Стэнли Кубрика по роману Стивена Кинга. История о писателе Джеке Торрансе, который становится смотрителем отеля Overlook зимой и постепенно сходит с ума. Фильм известен своей атмосферой, операторской работой и игрой Джека Николсона.

Экзорцист (1973) - культовый фильм ужасов режиссёра Уильяма Фридкина о девочке Риган, одержимой демоном. Считается одним из самых страшных фильмов всех времён. Известен шокирующими сценами и инновационными спецэффектами.

Хэллоуин (1978) - слэшер Джона Карпентера, который заложил основы жанра. История о маньяке Майкле Майерсе, который возвращается в родной город спустя 15 лет после убийства сестры. Создал образ 'неубиваемого' убийцы в маске.

Кошмар на улице Вязов (1984) - хоррор Уэса Крэйвена о Фредди Крюгере, который убивает подростков в их снах. Уникальная концепция страха перед сном. Джонни Депп дебютировал в этом фильме.

Пила (2004) - начало франшизы о маньяке Пазлголов, который заставляет жертв проходить жестокие испытания. Известен своими ловушками и неожиданными поворотами сюжета. Создал поджанр 'torture porn'.

Очень странные дела (2016-) - сериал Netflix, смешивающий хоррор, sci-fi и ностальгию по 80-м. История о детях, которые сталкиваются с паранормальными явлениями в городке Хокинс. Популяризировал ретро-эстетику в современном хорроре.

Наследственное (2018) - психологический хоррор Ари Астера о семье, столкнувшейся с тёмной тайной после смерти бабушки. Известен атмосферой нарастающего ужаса и игрой Тони Коллетт. Один из самых обсуждаемых хорроров последних лет.

Тихое место (2018) - постапокалиптический хоррор о семье, выживающей в мире, где охотятся существа, реагирующие на звук. Режиссёр и актёр - Джон Красински. Минималистичный саундтрек и напряжённая атмосфера."""

knowledge_file = "tmp/knowledge/horror_films.txt"
with open(knowledge_file, "w", encoding="utf-8") as f:
    f.write(horror_films_content)

import pathlib
file_path = pathlib.Path(knowledge_file).absolute()
knowledge_base.add_content(path=str(file_path))

search_agent = Agent(
    name="Advanced Search Agent",
    model=model,
    
    tools=[
        DuckDuckGoTools(enable_search=True, enable_news=True),
        ReasoningTools(
            add_instructions=True,  
        ),
    ],
    
    knowledge=knowledge_base,
    search_knowledge=True,
    
    db=db,
    
    enable_user_memories=True,
    
    add_history_to_context=True,
    num_history_runs=5,  
    
    instructions="""
    You are a horror film expert with a knowledge base and internet search access.
    
    SEARCH RULES:
    - For horror films: ONLY use search_knowledge (knowledge base)
    - For news, ratings, release dates: use DuckDuckGo
    - Knowledge base contains: The Shining, The Exorcist, Halloween, A Nightmare on Elm Street, Saw, Stranger Things, Hereditary, A Quiet Place
    
    CAPABILITIES:
    - Remember user preferences and watched films
    - Use ReasoningTools for planning responses
    - Give brief, informative answers (3-5 sentences)
    - Don't spoil without warning
    """,
    
    markdown=True,
    debug_mode=False,  
)

from IPython.display import display, Markdown

print("Эксперт по хоррор-фильмам")
print("Команды: 'выход', 'очистить память', 'показать память'\n")

user_id = "user_123"

existing_memories = search_agent.get_user_memories(user_id=user_id)
if existing_memories:
    display(Markdown(f"Найдено {len(existing_memories)} сохраненных воспоминаний\n"))

while True:
    user_query = input("Ты: ").strip()

    if not user_query:
        continue

    if user_query.lower() in ["выход", "exit", "quit"]:
        print("До встречи!")
        break

    if user_query.lower() in ["очистить память", "clear memory"]:
        try:
            db.clear_memories()
            print("Память очищена\n")
        except Exception as e:
            print(f"Ошибка: {e}\n")
        continue

    if user_query.lower() in ["показать память", "show memory"]:
        memories = search_agent.get_user_memories(user_id=user_id)
        if memories:
            display(Markdown(f"Воспоминания ({len(memories)}):"))
            for i, memory in enumerate(memories, 1):
                display(Markdown(f"{i}. {memory.memory}"))
            display(Markdown(""))
        else:
            print("Воспоминаний пока нет\n")
        continue

    display(Markdown(f"Ты:\n{user_query}"))

    response = search_agent.run(
        user_query,
        user_id=user_id,
        stream=False
    )

    display(Markdown(f"Агент:\n{response.content}"))

INFO Adding content from path, e497d74e-937e-5cfe-8677-b79d580e551c, None,                                         
     c:\Users\Ангелина\Desktop\agno_search_tasks\tmp\knowledge\horror_films.txt, None

WARNING  Contents DB not found for knowledge base

Эксперт по хоррор-фильмам
Команды: 'выход', 'очистить память', 'показать память'



Ты:
привет! я рыбок люблю и манго

Агент:
Привет! 😊 Рыбок и манго — интересное сочетание! 

Если вы хотите узнать что-то о фильмах ужасов или сериалах, я могу помочь с информацией из моей базы знаний. Для новостей и рейтингов использую поиск в интернете. Если у вас есть конкретный вопрос о фильме или сериале — спрашивайте!

Ты:
какие хоррор-фильмы посмотреть?

Агент:
Вот несколько отличных хоррор-фильмов для просмотра:

**Классика:**
- **The Shining** (1980) — Кубрик, безумие в изолированном отеле
- **The Exorcist** (1973) — Культовый фильм об изгнании демона
- **Halloween** (1978) — Истоки слэшер-жанра

**Современные:**
- **Hereditary** (2018) — Психологический ужас о семейном проклятии
- **A Quiet Place** (2018) — Триллер с уникальной концепцией (тишина = жизнь)

Если хотите что-то конкретное — экшн, психологию или сверхъестественное — уточните!

Ты:
а что я люблю, помнишь?

Агент:
Конечно помню! 😊 Ты любишь **рыбок** и **манго**. 

А что касается хоррор-фильмов — в нашей беседе ты ещё не упоминал, какие именно тебе нравятся. Если есть какие-то любимые — расскажи, и я учту это в рекомендациях!

До встречи!
